# Surrogate Factory — UCLoads
## Chapter 9. Model Validation
Objectives:
- **9.0** Validate the train/test split quality (voxel tesselation proximity method).
- **9.0b** Data coverage — distributions and PCA scatter across Train / Val / Test.
- **9.1** Predict structural loads on the test set.
- **9.2** Compute metrics (R², MAE, quantile90).
- **9.2b** KS distribution tests: train vs test residuals.
- **9.2c** Residual distribution plots.
- **9.3** Validate against requirements defined in SF_1.
- **9.4** Generate scatter and ratio plots.
- **9.5** Validation Report — full extended validation HTML report per model (validationlib template).
- **9.6** MLflow EDA Report.

### 0. Workflow initialisation

In [ ]:
import sys
from pathlib import Path

repo_root = str(Path('..').resolve().parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow('pipeline_config.yaml')
workflow.resume()

### 9. Model Validation

In [ ]:
workflow.import_metadata(stage_name='SF_9_Model_Validation')

In [ ]:
Train_set = workflow.load_data(workflow.config['job_name'] + '_Train_set.csv')
Val_set   = workflow.load_data(workflow.config['job_name'] + '_Val_set.csv')
Test_set  = workflow.load_data(workflow.config['job_name'] + '_Test_set.csv')
print(f'Train set : {Train_set.shape}')
print(f'Val set   : {Val_set.shape}')
print(f'Test set  : {Test_set.shape}')

#### 9.0 Split Validation
Checks that the train/test split is statistically sound:
- **Residual voxel proportion**: fraction of test points in bins not seen in training (target ≤ 0.05).
- **Phacking**: test points too close to training points (data leakage risk).
- **Isolated test**: test points too far from any training point (extrapolation).
- **Chi² p-value**: distribution of train vs test samples per voxel (target ≥ 0.05).

In [ ]:
from model_validation.split_val import split_validation
split_result = split_validation(workflow, Train_set, Test_set)

#### 9.0b Data Coverage — Train / Val / Test
Distribution of **inputs** and **outputs** across the three sets using validationlib.
- **doubleHistogram**: overlaid histograms for density comparison.
- **PCA scatter**: 2-D projection of 3-D input space coloured by set membership.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from validationlib.plots.hist import doubleHistogram
from validationlib.plots.cumu import doublecumulative

ms = workflow.metadata.get_step_data(['metadata', 'Model_Selection'])
inputs  = ms['inputs']
outputs = ms['outputs']

# All inputs are numeric
num_inputs = inputs
print(f'Inputs: {num_inputs}')

fig = doubleHistogram(
    Train_set[num_inputs], Test_set[num_inputs],
    x1label='Train', x2label='Test', xlabel='Input features',
    multiPlotsKwargs={'figHsize': 14, 'figAspectRatio': 3}
)
fig.suptitle('Input distribution — Train vs Test', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig)

fig2 = doubleHistogram(
    Train_set[outputs], Test_set[outputs],
    x1label='Train', x2label='Test', xlabel='Outputs',
    multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
)
fig2.suptitle('Output distribution — Train vs Test', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig2)

# PCA scatter (3 inputs → 2 PCs)
all_num = pd.concat([Train_set[num_inputs], Val_set[num_inputs], Test_set[num_inputs]], axis=0)
pca = PCA(n_components=2)
pca.fit(all_num)
trPC = pca.transform(Train_set[num_inputs])
vlPC = pca.transform(Val_set[num_inputs])
tsPC = pca.transform(Test_set[num_inputs])

fig3, ax = plt.subplots(figsize=(8, 6))
ax.scatter(trPC[:,0], trPC[:,1], s=8,  alpha=0.4, label=f'Train (n={len(Train_set)})',  color='steelblue')
ax.scatter(vlPC[:,0], vlPC[:,1], s=12, alpha=0.6, label=f'Val   (n={len(Val_set)})',    color='orange',    marker='s')
ax.scatter(tsPC[:,0], tsPC[:,1], s=12, alpha=0.6, label=f'Test  (n={len(Test_set)})',   color='tomato',    marker='^')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Train / Val / Test coverage — PCA of inputs')
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig3)

#### 9.1 Predictions

In [ ]:
from model_validation.prediction import predict
model_output = predict(workflow, Test_set)
train_output = predict(workflow, Train_set)
val_output   = predict(workflow, Val_set)
model_output.head()

#### 9.2 Metrics

In [ ]:
from model_validation.score import calculate_metrics
metrics = calculate_metrics(workflow, Test_set, model_output)
JSON(metrics)

#### 9.2b Distribution Tests (KS: train vs test residuals)
Kolmogorov-Smirnov test comparing residual distributions on **training** vs **test** set.
- H₀: both distributions are the same.
- p-value ≥ 0.05 → ✓ healthy generalisation.
- p-value < 0.05 → ✗ possible overfitting or distribution shift.

In [ ]:
from model_validation.score import distribution_tests
ks_results = distribution_tests(workflow, Train_set, Test_set, train_output, model_output)

#### 9.2c Residual Distribution Plots

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from validationlib.plots.hist import doubleHistogram
from validationlib.plots.cumu import doublecumulative

ms = workflow.metadata.get_step_data(['metadata', 'Model_Selection'])
outputs = ms['outputs']
models_info = workflow.metadata.get_step_data(['metadata', 'Model_Training', 'Models'])

for info in models_info:
    label = info['label']

    train_res = pd.DataFrame(
        {col: Train_set[col].values - train_output[f'{label}__{col}'].values for col in outputs}
    )
    test_res = pd.DataFrame(
        {col: Test_set[col].values - model_output[f'{label}__{col}'].values for col in outputs}
    )

    fig = doubleHistogram(
        train_res, test_res,
        x1label='Train residuals', x2label='Test residuals', xlabel='Outputs',
        multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
    )
    fig.suptitle(f'{label} — Residual Distribution (Train vs Test)', y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    fig2 = doublecumulative(
        train_res, test_res,
        label1='Train', label2='Test', xlabel='Residuals',
        multiPlotsKwargs={'figHsize': 18, 'figAspectRatio': 3}
    )
    fig2.suptitle(f'{label} — Cumulative Residuals (Train vs Test)', y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig2)

#### 9.3 Validation against requirements

In [ ]:
from model_validation.validation import validate
validate(workflow, metrics)

#### 9.3b Validation Summary — Interactive Table

In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

validation_results = workflow.metadata.get_step_data(['metadata', 'Model_Validation', 'validation_results'])
models_info_w = workflow.metadata.get_step_data(['metadata', 'Model_Training', 'Models'])
labels_w = [m['label'] for m in models_info_w]

rows = []
for vr in validation_results:
    row = {'Output': vr['output'], 'Metric': vr['metric'], 'Target': f"< {vr['target']}"}
    for lbl in labels_w:
        m = vr['models'].get(lbl, {})
        score, passed = m.get('score'), m.get('passed')
        row[lbl] = (f"{'✅' if passed else '❌'} {score:.4f}") if score is not None else '?'
    rows.append(row)

df_val = pd.DataFrame(rows).set_index('Output')

def color_cell(val):
    if '✅' in str(val): return 'background-color:#d4edda; color:#155724'
    if '❌' in str(val): return 'background-color:#f8d7da; color:#721c24'
    return ''

styled = df_val.style.map(color_cell, subset=labels_w).set_caption('Validation Results')
display(styled)

#### 9.4 Plots

In [ ]:
%matplotlib inline
from model_validation.visualize import plot
plot(workflow, Test_set, model_output)

#### 9.4b Training History
Reloads each saved model and plots its training history — loss per iteration,
plus the validation score for models trained with early stopping. Saved as
`artifacts/training_curve.png` and reused by the executive-summary report.


In [ ]:
%matplotlib inline
from model_validation.training_curve import plot_training_curve
from IPython.display import Image, display

curve_path = plot_training_curve(workflow)
if curve_path:
    display(Image(filename=curve_path))


#### 9.4b Interactive Scatter Explorer

In [ ]:
%matplotlib inline
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

ms_w      = workflow.metadata.get_step_data(["metadata", "Model_Selection"])
outputs_w = ms_w["outputs"]
mi_w      = workflow.metadata.get_step_data(["metadata", "Model_Training", "Models"])
labels_w  = [m["label"] for m in mi_w]

model_dd  = widgets.Dropdown(options=labels_w,  description="Model:",  layout=widgets.Layout(width="180px"))
output_dd = widgets.Dropdown(options=outputs_w, description="Output:", layout=widgets.Layout(width="260px"))
out_w     = widgets.Output()

def update_scatter(*_):
    label = model_dd.value
    col   = output_dd.value
    with out_w:
        out_w.clear_output(wait=True)
        y_true = Test_set[col].values
        y_pred = model_output[f"{label}__{col}"].values
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        lo, hi = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
        axes[0].plot([lo, hi], [lo, hi], "k--", lw=0.8, label="y = x")
        axes[0].scatter(y_true, y_pred, s=8, alpha=0.45, color="steelblue")
        axes[0].set_xlabel("True"); axes[0].set_ylabel("Predicted")
        axes[0].set_title(f"{label} — {col}  (predicted vs true)")
        axes[0].legend(fontsize=8)
        mask  = y_true != 0
        ratio = y_pred[mask] / y_true[mask]
        axes[1].axhline(1.0, color="k", lw=0.8, linestyle="--", label="ratio = 1")
        axes[1].scatter(y_true[mask], ratio, s=8, alpha=0.45, color="tomato")
        axes[1].set_xlabel("True"); axes[1].set_ylabel("y_pred / y_true")
        axes[1].set_title(f"{label} — {col}  (ratio)")
        axes[1].legend(fontsize=8)
        plt.tight_layout()
        plt.show()

model_dd.observe(update_scatter, names="value")
output_dd.observe(update_scatter, names="value")
update_scatter()
display(widgets.HBox([model_dd, output_dd]), out_w)

#### 9.5 Validation Report
Exports CSVs per model and runs `validation_script.py` to produce a full HTML validation report.
Reports are saved to `data/artifacts/validation_reports/`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
from model_validation.export_validation_csvs import export_validation_csvs

csv_dirs = export_validation_csvs(
    workflow,
    Train_set, Val_set, Test_set,
    train_output, val_output, model_output,
)

script_path = Path(workflow.config["data.folder"]).parent / "python_nodes_library" / "model_validation" / "validation_script.py"
output_dir  = Path(workflow.config["artifacts.folder"]) / "validation_reports"
output_dir.mkdir(parents=True, exist_ok=True)

ms_w = workflow.metadata.get_step_data(["metadata", "Model_Selection"])
num_inputs_w = Train_set[ms_w["inputs"]].select_dtypes(include="number").columns.tolist()

# Repo root = .../<repo>/<UseCase>/pipeline/python_nodes_library/model_validation -> up 4.
# Derived, never hardcoded, so the notebook runs on any machine.
repo_root = script_path.parents[4]
env = {
    **os.environ,
    "PYTHONPATH": str(repo_root / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
    "MLFLOW_ALLOW_FILE_STORE": "true",
}

# The report template runs voxel tesselation + convex-hull membership, which is
# O(n^2) in the number of unique input combinations. Above a few thousand rows
# it never finishes, so cap the sample the template works on.
subsample = 300 if len(Train_set) > 5000 else None
print(f"Train rows: {len(Train_set):,} -> subsample_size = {subsample or 'full set'}")

for label, csv_dir in csv_dirs.items():
    print(f"Running validation report for {label}...")
    cmd = [
        sys.executable, str(script_path),
        "-d", str(csv_dir), "-n", label, "-o", str(output_dir),
        "--exclude_warnings",
    ]
    if subsample:
        cmd += ["--subsample_size", str(subsample)]
    cmd += ["--splitting_variables", *num_inputs_w]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800, env=env)
        if result.returncode == 0:
            print(f"  OK  {output_dir / (label + '_validation_output.html')}")
        else:
            print(f"  FAILED\n{result.stderr[-2000:]}")
    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT (>1800s) for {label} - lower subsample further.")


#### 9.6 MLflow EDA Report
Logs metrics, distribution plots, PCA coverage and split quality to MLflow.

In [ ]:
from tracking.mlflow_eda import log_eda

log_eda(
    workflow    = workflow,
    Train_set   = Train_set,
    Val_set     = Val_set,
    Test_set    = Test_set,
    model_output  = model_output,
    train_output  = train_output,
    metrics       = metrics,
    split_result  = split_result,
    ks_results    = ks_results,
)

### Save

In [ ]:
workflow.save_metadata()

#### 9.7 Executive Summary Report (PDF + HTML)

Generates the mandatory multi-part validation report via
`reporting/generate_executive_summary.py`:

- **Part 1 - Executive Summary** (winner model, <= 2 pages)
- **Part 2 - Technical Review** (all models: TOC, scatter/ratio, KS, split, roadmap)
- **Part 3 - Deep Analysis** (winner model: data overview, train-test split,
  error quantification, P(E|X), P(E|Y), uncertainty)

Output lands in `data/artifacts/validation_reports/`.

> **Needs a LaTeX engine.** `tectonic` is preferred but is **not available via pip**.
> Install the static binary (no root, no conda):
> ```
> mkdir -p ~/.local/bin && cd ~/.local/bin
> curl --proto "=https" --tlsv1.2 -fsSL https://drop-sh.fullyjustified.net | sh
> export PATH="$HOME/.local/bin:$PATH"
> ```
> Any TeX Live install also works - the report falls back to
> `latexmk`, `xelatex` or `pdflatex` automatically.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

pipeline_dir  = Path(workflow.config["data.folder"]).parent
artifacts_dir = Path(workflow.config["artifacts.folder"])
job           = workflow.config["job_name"]

report_script = pipeline_dir / "python_nodes_library" / "reporting" / "generate_executive_summary.py"
meta_json     = artifacts_dir / f"metadata_{job}.json"
report_dir    = artifacts_dir / "validation_reports"
report_dir.mkdir(parents=True, exist_ok=True)

# Repo root = .../<repo>/<UseCase>/pipeline -> up 2. Keeps this portable across machines.
repo_root = pipeline_dir.parent.parent
env = {
    **os.environ,
    "PYTHONPATH": str(repo_root / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
    "MLFLOW_ALLOW_FILE_STORE": "true",
}

engines = [e for e in ("tectonic", "latexmk", "xelatex", "pdflatex") if shutil.which(e)]
print(f"LaTeX engines available: {', '.join(engines) if engines else 'NONE'}")

if not report_script.exists():
    print(f"Report script not found: {report_script}")
elif not meta_json.exists():
    print(f"Metadata not found: {meta_json}\n   Run the 'Save' cell above first.")
else:
    try:
        result = subprocess.run(
            [sys.executable, str(report_script), str(meta_json), "--output", str(report_dir)],
            capture_output=True, text=True, timeout=1800, env=env,
        )
        if result.stdout:
            print(result.stdout[-4000:])
        if result.returncode == 0:
            for f in sorted(report_dir.glob("executive_summary_*")) + \
                     sorted(report_dir.glob("deep_analysis_*")):
                print(f"  OK  {f.name}  ({f.stat().st_size/1024:.0f} KB)")
        else:
            print("Report generation failed:")
            print(result.stderr[-3000:])
    except subprocess.TimeoutExpired:
        print("Timeout (>1800s) building the executive summary report.")
